# 02 — Silver Clean

Read **bronze**, enforce the canonical schema from `schemas/bank_schemas.py`, fix raw-source quirks, deduplicate on the primary key, and write **silver** Delta tables.

## Cleaning rules applied here

| Issue | Fix |
|---|---|
| `customer_id` arrives as `BIGINT` from JSON loans | Cast to `STRING` and **format as `C{:03d}`** to match the rest |
| `payments` has no `customer_id` | Derive `sender_customer_id` by joining bronze `bank_accounts` |
| All CSV columns came in as `STRING` | Cast each column to its canonical type from `bank_schemas` |
| Duplicate rows in `customers` (we injected one) | Drop duplicates on the primary key |
| Empty strings from CSV (`paid_date`, `paid_amount`, `credit_limit`, `email`) | Convert empty string → NULL before casting |
| Bronze metadata columns (`_ingest_ts`, `_source_file`) | Drop — silver is the curated layer, provenance lives in bronze |
| Rows missing a primary-key value | Filter out |

## Setup

In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run from inside project/ or repo root")
sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from conf.spark import get_spark, BRONZE_DIR, SILVER_DIR
from schemas.bank_schemas import SILVER_SCHEMAS, PRIMARY_KEYS

spark = get_spark("SilverClean")
SILVER_DIR.mkdir(parents=True, exist_ok=True)
print(f"BRONZE: {BRONZE_DIR}")
print(f"SILVER: {SILVER_DIR}")

In [ ]:
def read_bronze(table: str):
    return spark.read.format("delta").load(str(BRONZE_DIR / table)) \
                .drop("_ingest_ts", "_source_file")

def empty_to_null(df, *cols):
    """Convert empty-string columns to NULL — common quirk of CSV."""
    for c in cols:
        df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)))
    return df

def write_silver(df, table: str):
    schema = SILVER_SCHEMAS[table]
    pk = PRIMARY_KEYS[table]
    # Cast each column to its canonical type, dropping anything outside the schema
    df = df.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields])
    df = df.filter(F.col(pk).isNotNull()).dropDuplicates([pk])
    target = SILVER_DIR / table
    (df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").save(str(target)))
    n = spark.read.format("delta").load(str(target)).count()
    print(f"  silver.{table:24s}  {n:>5} rows")

## customers — straightforward, just dedupe and empty→null

In [ ]:
df = read_bronze("customers")
df = empty_to_null(df, "email", "phone", "date_of_birth", "gender")
write_silver(df, "customers")

## card_accounts and card_transactions

In [ ]:
df = read_bronze("card_accounts")
df = empty_to_null(df, "credit_limit")
write_silver(df, "card_accounts")

df = read_bronze("card_transactions")  # already typed (Parquet source)
write_silver(df, "card_transactions")

## loan_accounts and loan_repayments — cast `customer_id` from BIGINT to canonical `C{:03d}`

This is the cross-source standardisation step the design note in `02-data-model.ipynb` calls out.

In [ ]:
def fix_loan_customer_id(df):
    return df.withColumn(
        "customer_id",
        F.concat(F.lit("C"), F.lpad(F.col("customer_id").cast("string"), 3, "0"))
    )

df = read_bronze("loan_accounts")
df = fix_loan_customer_id(df)
write_silver(df, "loan_accounts")

df = read_bronze("loan_repayments")
df = fix_loan_customer_id(df)
df = empty_to_null(df, "paid_date", "paid_amount")
write_silver(df, "loan_repayments")

## bank_accounts and account_transactions

In [ ]:
df = read_bronze("bank_accounts")
df = empty_to_null(df, "interest_rate")
write_silver(df, "bank_accounts")

df = read_bronze("account_transactions")
write_silver(df, "account_transactions")

## payments — derive `sender_customer_id` by joining `bank_accounts`

Raw payments only know `sender_account_id` and `receiver_account_id`. Silver enriches by looking up the owning customer of the sender account.

In [ ]:
payments_raw = read_bronze("payments")
ba = (spark.read.format("delta").load(str(SILVER_DIR / "bank_accounts"))
        .select(F.col("account_id").alias("sender_account_id"),
                F.col("customer_id").alias("sender_customer_id")))

before = payments_raw.count()
df = (payments_raw.join(ba, on="sender_account_id", how="left"))
missing = df.filter(F.col("sender_customer_id").isNull()).count()
print(f"payments rows: {before}, missing sender_customer_id after join: {missing}")

write_silver(df, "payments")

## Verify: prove `customer_id` is now uniformly STRING `Cxxx` everywhere

In [ ]:
for t in ["customers", "card_accounts", "card_transactions",
          "loan_accounts", "loan_repayments",
          "bank_accounts", "account_transactions"]:
    df = spark.read.format("delta").load(str(SILVER_DIR / t))
    sample = [r[0] for r in df.select("customer_id").limit(3).collect()]
    print(f"{t:24s}  {df.schema['customer_id'].dataType.simpleString():8s}  sample={sample}")

p = spark.read.format("delta").load(str(SILVER_DIR / "payments"))
samp = [r[0] for r in p.select("sender_customer_id").limit(3).collect()]
print(f"{'payments':24s}  {p.schema['sender_customer_id'].dataType.simpleString():8s}  sample={samp}  (sender_customer_id)")

In [ ]:
# Quick row-count summary
from pyspark.sql import Row
rows = []
for t in SILVER_SCHEMAS:
    n = spark.read.format("delta").load(str(SILVER_DIR / t)).count()
    rows.append(Row(table=t, rows=n))
spark.createDataFrame(rows).orderBy("table").show(truncate=False)

## Summary

Silver tables are typed, deduped, and use a uniform `customer_id` STRING. Continue with **03-gold-marts.ipynb**.